# Task 8: Semantic Segmentation Pipeline on Pixel-wise Grids using custom Dice Loss

**Objective:** Build a spatial localization engine by coding a U-Net architecture, loading target masks using Albumentations augmentations, and implementing a composite BCE + Dice overlap loss from scratch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Custom Soft Dice Loss + Binary Cross Entropy Loss
class DiceBCELoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
        
    def forward(self, pred_logits, targets):
        # Sigmoid to normalize outputs to probabilities
        probs = torch.sigmoid(pred_logits)
        
        # Flatten prediction and label vectors
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)
        
        # BCE Loss
        bce = F.binary_cross_entropy_with_logits(pred_logits, targets, reduction='mean')
        
        # Dice Coefficient Calculation
        intersection = (probs_flat * targets_flat).sum()
        union = probs_flat.sum() + targets_flat.sum()
        dice_loss = 1.0 - (2.0 * intersection + self.eps) / (union + self.eps)
        
        return bce + dice_loss

# 2. Simple U-Net Model Implementation
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(64, 128)
        self.upconv = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec = DoubleConv(128, 64)
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)
        
    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        bn = self.bottleneck(self.pool(x2))
        
        up = self.upconv(bn)
        dec_in = torch.cat([up, x2], dim=1)
        out = self.dec(dec_in)
        # Skip connections can be added similarly
        return self.final(out)

In [ ]:
# Synthetic segmentation dataset
class SyntheticSegmentationDataset(Dataset):
    def __init__(self, size=64, transform=None):
        self.size = size
        self.transform = transform
        
    def __len__(self):
        return self.size
        
    def __getitem__(self, idx):
        # Generate random image with circle mask shapes
        img = np.zeros((128, 128, 3), dtype=np.uint8)
        mask = np.zeros((128, 128), dtype=np.uint8)
        
        # Draw circle
        cy, cx = np.random.randint(30, 90, size=2)
        radius = np.random.randint(15, 30)
        cv2.circle(img, (cx, cy), radius, (255, 255, 255), -1)
        cv2.circle(mask, (cx, cy), radius, 1, -1)
        
        # Add noise
        img = (img + np.random.randint(0, 50, img.shape)).clip(0, 255).astype(np.uint8)
        
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
            
        return img, mask.float().unsqueeze(0)

# Transforms using Albumentations
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2()
])

dataset = SyntheticSegmentationDataset(size=64, transform=transform)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

model = UNet()
criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Single train epoch verification
model.train()
for img, mask in dataloader:
    optimizer.zero_grad()
    pred = model(img)
    loss = criterion(pred, mask)
    loss.backward()
    optimizer.step()
    
print(f"Training successfully verified! Single-batch Loss: {loss.item():.4f}")